# Day 1 — Exploratory Data Analysis (EDA)

## Customer Churn Dataset

This notebook performs exploratory data analysis and documents the data-quality checks and preprocessing required for the Day 1 machine-learning exercise.

### Objectives
- Load the raw dataset from `data/raw`
- Inspect the dataset structure and quality
- Analyze missing values, duplicates, and outliers
- Clean the dataset
- Analyze the target variable and class imbalance
- Visualize numerical feature distributions
- Analyze correlations
- Analyze categorical features against churn
- Produce the cleaned dataset in `data/processed/final.csv`


## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

plt.rcParams["figure.figsize"] = (8, 5)


## 2. Load the Raw Dataset

In [2]:
RAW_PATH = Path("../data/raw/raw_dataset.csv")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()


Dataset loaded successfully.
Shape: (1512, 13)


,customer_id,age,income,tenure_years,monthly_spend,support_calls,satisfaction_score,login_frequency,contract_months,city,plan,payment_method,churn
0,CUST100908,64.0,60725.55,10.0,55.54,1.0,6.99,22.21,6.0,Jaipur,Premium,UPI,0
1,CUST100618,61.0,52169.75,15.0,68.82,3.0,7.93,22.87,12.0,Mumbai,Premium,Card,0
2,CUST101387,48.0,62551.90,6.0,77.05,2.0,4.61,30.41,12.0,Pune,Standard,Card,0
3,CUST100942,39.0,31013.15,10.0,68.71,2.0,8.07,23.51,36.0,Delhi,Premium,Card,0
4,CUST100304,33.0,25891.10,15.0,62.07,2.0,4.68,16.07,24.0,Delhi,Premium,UPI,0


## 3. Basic Dataset Inspection

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

print("\nDescriptive statistics:")
df.describe()


## 4. Missing-Value Analysis

First, missing values are inspected in the original dataset before any imputation is performed.

In [ ]:
missing_counts = df.isnull().sum()
missing_percentages = df.isnull().mean() * 100

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percentage": missing_percentages
})

missing_summary


In [ ]:
# Missing-value heatmap using the original raw dataset
plt.figure(figsize=(10, 6))
plt.imshow(df.isnull(), aspect="auto")

plt.title("Missing Values Heatmap")
plt.xlabel("Columns")
plt.ylabel("Rows")

plt.xticks(
    range(len(df.columns)),
    df.columns,
    rotation=45,
    ha="right"
)

plt.tight_layout()
plt.savefig(PROCESSED_DIR / "missing_value_heatmap.png")
plt.show()
plt.close()


## 5. Define Numerical Features

In [ ]:
numerical_columns = [
    "age",
    "income",
    "tenure_years",
    "monthly_spend",
    "support_calls",
    "satisfaction_score",
    "login_frequency",
    "contract_months"
]

categorical_columns = [
    "city",
    "payment_method"
]


## 6. Outlier Analysis

Two standard approaches are used:

- **IQR method:** values outside `Q1 - 1.5 × IQR` and `Q3 + 1.5 × IQR`
- **Z-score method:** absolute Z-score greater than 3


In [ ]:
def detect_iqr_outliers(dataframe, column):
    q1 = dataframe[column].quantile(0.25)
    q3 = dataframe[column].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    mask = (
        (dataframe[column] < lower_bound) |
        (dataframe[column] > upper_bound)
    )

    return mask, lower_bound, upper_bound


def detect_zscore_outliers(dataframe, column):
    z_scores = (
        (dataframe[column] - dataframe[column].mean())
        / dataframe[column].std()
    )
    return np.abs(z_scores) > 3


outlier_results = []

for column in numerical_columns:
    iqr_mask, lower_bound, upper_bound = detect_iqr_outliers(df, column)
    zscore_mask = detect_zscore_outliers(df, column)

    outlier_results.append({
        "feature": column,
        "iqr_lower_bound": lower_bound,
        "iqr_upper_bound": upper_bound,
        "iqr_outliers": int(iqr_mask.sum()),
        "zscore_outliers": int(zscore_mask.sum())
    })

outlier_summary = pd.DataFrame(outlier_results)
outlier_summary


In [ ]:
# Inspect IQR outlier values
for column in numerical_columns:
    outliers, _, _ = detect_iqr_outliers(df, column)

    print(f"\n{column}:")
    print(df.loc[outliers, column].sort_values().to_list())


## 7. Remove Confirmed Artificial Outliers

The confirmed artificial values identified during the analysis are removed. Natural extreme values are not removed solely because they are statistically unusual.

In [ ]:
confirmed_outliers = {
    "income": [350000, 420000, 500000],
    "monthly_spend": [900, 1100],
    "support_calls": [18, 22, 25]
}

original_rows = len(df)

df = df[
    ~df["income"].isin(confirmed_outliers["income"]) &
    ~df["monthly_spend"].isin(confirmed_outliers["monthly_spend"]) &
    ~df["support_calls"].isin(confirmed_outliers["support_calls"])
].reset_index(drop=True)

removed_rows = original_rows - len(df)

print("Original rows:", original_rows)
print("Removed artificial outlier rows:", removed_rows)
print("Remaining rows:", len(df))


## 8. Missing-Value Imputation

Missing numerical values are filled with the **median**, while missing categorical values are filled with the **mode**.

In [ ]:
# Numerical imputation
for column in ["age", "income", "satisfaction_score"]:
    df[column] = df[column].fillna(df[column].median())

# Categorical imputation
for column in ["city", "payment_method"]:
    df[column] = df[column].fillna(df[column].mode()[0])

print("Missing values after imputation:")
print(df.isnull().sum())


## 9. Duplicate Analysis and Removal

In [ ]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows before removal:", duplicate_count)

df = df.drop_duplicates().reset_index(drop=True)

print("Duplicate rows after removal:", df.duplicated().sum())
print("Final cleaned shape:", df.shape)


## 10. Save the Final Processed Dataset

In [ ]:
FINAL_PATH = PROCESSED_DIR / "final.csv"

df.to_csv(FINAL_PATH, index=False)

print(f"Final dataset saved to: {FINAL_PATH}")
print("Final shape:", df.shape)


## 11. Target Distribution

The target variable is `churn`.

The distribution is examined both as counts and percentages to identify class imbalance.

In [ ]:
target_counts = df["churn"].value_counts().sort_index()
target_percentages = df["churn"].value_counts(normalize=True).sort_index() * 100

target_summary = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentages
})

target_summary


In [ ]:
target_counts.plot(kind="bar")

plt.title("Target Distribution — Churn")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)

plt.tight_layout()
plt.savefig(PROCESSED_DIR / "target_distribution.png")
plt.show()
plt.close()


## 12. Numerical Feature Distributions

In [ ]:
for column in numerical_columns:
    df[column].hist(bins=30)

    plt.title(f"Distribution of {column}")
    plt.xlabel(column)
    plt.ylabel("Frequency")

    plt.tight_layout()
    plt.savefig(PROCESSED_DIR / f"{column}_distribution.png")
    plt.show()
    plt.close()


## 13. Outlier Visualization

In [ ]:
for column in numerical_columns:
    plt.figure(figsize=(8, 5))

    # Use the cleaned data to inspect the remaining distribution.
    plt.boxplot(df[column].dropna())

    plt.title(f"Boxplot of {column}")
    plt.ylabel(column)

    plt.tight_layout()
    plt.savefig(PROCESSED_DIR / f"{column}_boxplot.png")
    plt.show()
    plt.close()


## 14. Correlation Analysis

In [ ]:
correlation_matrix = df[numerical_columns + ["churn"]].corr()

correlation_matrix


In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(correlation_matrix, aspect="auto")

plt.colorbar()

plt.xticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns
)

for i in range(len(correlation_matrix)):
    for j in range(len(correlation_matrix)):
        plt.text(
            j,
            i,
            f"{correlation_matrix.iloc[i, j]:.2f}",
            ha="center",
            va="center"
        )

plt.title("Correlation Matrix")
plt.tight_layout()
plt.savefig(PROCESSED_DIR / "correlation_heatmap.png")
plt.show()
plt.close()


## 15. Categorical Feature Analysis

Churn rates are compared across the categorical features `city` and `payment_method`.

These results describe associations in the dataset and do not imply causation.

In [ ]:
for column in categorical_columns:
    print(f"\nChurn by {column}:")
    print(
        pd.crosstab(
            df[column],
            df["churn"],
            normalize="index"
        ) * 100
    )


In [ ]:
for column in categorical_columns:
    churn_rate = (
        df.groupby(column)["churn"]
        .mean()
        .sort_values(ascending=False)
    )

    churn_rate.plot(kind="bar")

    plt.title(f"Churn Rate by {column}")
    plt.xlabel(column)
    plt.ylabel("Churn Rate")

    plt.tight_layout()
    plt.savefig(PROCESSED_DIR / f"{column}_churn_rate.png")
    plt.show()
    plt.close()


## 16. EDA Conclusions

### Data quality
- Missing values were identified in the raw dataset and handled using median imputation for numerical variables and mode imputation for categorical variables.
- Duplicate records were identified and removed.
- Confirmed artificial outliers were removed after investigation rather than deleting every statistically unusual observation.

### Target variable
The cleaned dataset contains approximately **73.6% non-churned customers and 26.4% churned customers**, indicating class imbalance. This should be considered when training the machine-learning model.

### Numerical features
The strongest observed linear relationships with churn are:
- `age`: positive relationship with churn.
- `satisfaction_score`: negative relationship with churn.
- `income`: weaker negative relationship with churn.

Several other numerical variables have weak or near-zero linear correlations with churn. Weak correlation does not necessarily mean a feature has no predictive value because non-linear relationships and interactions may still exist.

### Categorical features
Churn rates differ across cities and payment methods. These differences indicate that categorical features may contain useful predictive information and should be retained for subsequent modeling.

### Next steps
The cleaned dataset has been saved as `data/processed/final.csv`. The next stage can use this dataset for feature preprocessing, train/validation/test splitting, scaling, class-imbalance handling, and model development.
